# Python to SQL, and back again
In this codealong we will show you how to create a relational database from your pandas DataFrames.
> **To run this notebook you will need to work locally and not on colab.**

---
## 1.&nbsp; Import libraries 💾
If you haven't already installed sqlalchemy, you will need to. Uncomment the code below, install, and then recomment the code - you only need to install it once.

In [13]:
# install if needed
# !conda install sqlalchemy
# !conda install pymysql

In [14]:
import pandas as pd

In [25]:
pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


---
## 2.&nbsp; Relational Databases 📂

Creating DataFrames in python and pandas often results in tables with repeated information, as shown in the example below.
<br>

| author_name | book_title | year_published |
| --- | --- | --- |
| Arthur Conan Doyle | The Adventures of Sherlock Holmes | 1887 |
| J.R.R. Tolkien | The Hobbit | 1937 |
| J.R.R. Tolkien | The Lord of the Rings | 1954 |
| Harper Lee | To Kill a Mockingbird | 1960 |
| Harper Lee | Go Set a Watchman | 2015 |
<br>

This can be problematic for relational databases, which are designed to store data efficiently and avoid redundancy. To address this issue, we will separate the author and book information into two tables: authors and books. This approach eliminates duplicate data, ensuring data integrity and optimising storage.
<br>

| author_id | author_name |
| --- | --- |
| 1 | Arthur Conan Doyle |
| 2 | J.R.R. Tolkien |
| 3 | Harper Lee |
<br>

| book_id | book_title | year_published | author_id |
|---|---|---|---|
| 1 | The Adventures of Sherlock Holmes | 1887 | 1 |
| 2 | The Hobbit | 1937 | 2 |
| 3 | The Lord of the Rings | 1954 | 2 |
| 4 | To Kill a Mockingbird | 1960 | 3 |
| 5 | Go Set a Watchman | 2015 | 3 |

---
## 3.&nbsp; Creating the authors table with python 🐍
Let's start by creating the original DataFrame, including the repeated data.

In [26]:
names = ["Arthur Conan Doyle", "J.R.R. Tolkien", "J.R.R. Tolkien", "Harper Lee", "Harper Lee"]
titles = ["The Adventures of Sherlock Holmes", "The Hobbit", "The Lord of the Rings", "To Kill a Mockingbird", "Go Set a Watchman"]
years = [1887, 1937, 1954, 1960, 2015]

non_relational_df = pd.DataFrame({"author_name": names,
                                  "book_title": titles,
                                  "year_published": years})

non_relational_df

,author_name,book_title,year_published
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887
1,J.R.R. Tolkien,The Hobbit,1937
2,J.R.R. Tolkien,The Lord of the Rings,1954
3,Harper Lee,To Kill a Mockingbird,1960
4,Harper Lee,Go Set a Watchman,2015


Now, let's select only the authors without any duplicates.

In [27]:
authors_unique = non_relational_df["author_name"].unique()

authors_df = pd.DataFrame({"author_name": authors_unique})

authors_df

,author_name
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


Fantastic! This DataFrame will be the foundation of our authors table.

---
## 4.&nbsp; Creating the matching authors table with SQL 💻

Ok, now we're ready to store this DataFrame in SQL. Before we can send the information in SQL, we need to make a table that has the same columns and data types to recieve the data. While we are creating a table for authors, we can also create the books table too.

Open MySQL Workbench, open a local connection, and open a new file. Then copy and paste the code from below.

```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS sql_workshop ;

-- Create the database
CREATE DATABASE sql_workshop;

-- Use the database
USE sql_workshop;

-- Create the 'authors' table
CREATE TABLE authors (
    author_id INT AUTO_INCREMENT, -- Automatically generated ID for each author
    author_name VARCHAR(255) NOT NULL, -- Name of the author
    PRIMARY KEY (author_id) -- Primary key to uniquely identify each author
);

-- Create the 'books' table
CREATE TABLE books (
    book_id INT AUTO_INCREMENT, -- Automatically generated ID for each book
    book_title VARCHAR(255) NOT NULL, -- Title of the book
    year_published INT, -- Year the book was published
    author_id INT, -- ID of the author who wrote the book
    PRIMARY KEY (book_id), -- Primary key to uniquely identify each book
    FOREIGN KEY (author_id) REFERENCES authors(author_id) -- Foreign key to connect each book to its author
);
```

To download the sql file that we will follow for this section, [click here](https://drive.google.com/uc?export=download&id=1tln_33FM7D9wLckzxacBJNcMYqtyxybE)

If you'd like more information about MySQL data types [click here](https://www.w3schools.com/mysql/mysql_datatypes.asp).

---
## 5.&nbsp; Sending the information from this notebook to sql 📠
To establish a connection with the SQL database, we need to provide the notebook with the necessary information, which we do using the connection string below. You will need to modify only the password variable, which should match the password you set during MySQL Workbench installation.

### 5.1. Method 1: Direct coding

You can write and run the following code block directly in your Notebook or Python script.

**Pros**: Direct, easy      
**Cons**: Need to clean Notebook(s) or scripts before sharing anywhere

In [28]:
schema = "gans"
host = "127.0.0.1" # or "localhost"
user = "root"
password = "YOUR_PASSWORD"
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

### 5.2. Method 2: User-entered password

You can replace directly writing your password in the code with an input.

**Pros**: Password stays hidden, no need to clean Notebook(s) or Python script(s)      
**Cons**: Must enter password anytime the code runs, not suitable for full automation

In [29]:
import getpass

schema = "gans"
host = "127.0.0.1"
user = "root"
password = getpass.getpass("Enter password")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

Enter password ········


### 5.3. Method 3: Use a module

The connection string can be coded into an external file and accessed via `import`

**Pros**: Write once and use in many Notebooks or Python scripts, modifiable to automate     
**Cons**: Plain text (not secure), bad for large organizations

In [30]:
import con_EXAMPLE

con_EXAMPLE.connection_string

ModuleNotFoundError: No module named 'con_EXAMPLE'

### 5.4. Method 4: Create a `.env` file

Environment files are often used to store and share information used by multiple code files or, because the library used to read them travels "up" a directory tree until it finds a `.env` file, even multiple projects.


**Pros**: Write once and use in many Notebooks or Python scripts, mirrors setups used on many cloud platforms     
**Cons**: Plain text (not secure), bad for large organizations

In [31]:
import os
from dotenv import load_dotenv

load_dotenv()
os.getenv("CON_STRING")

To send information to our sql databse we use the pandas method `.to_sql()`. The argument `if_exists="append"` says that we don't want to overwrite any existing data, but add on to what is already there.

In [32]:
authors_df.to_sql('authors',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

Now, have a look at the table `authors` in MySQL Workbench, you should see that the names of the authors have appeared.

---
## 6.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [33]:
authors_from_sql = pd.read_sql("authors", con=connection_string)
authors_from_sql

,author_name
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


Using this same method, we can also perform SQL queries to only bring back certain sections of information instead of the whole DataFrame.

In [34]:
pd.read_sql("""
            SELECT DISTINCT author_name
            FROM authors
            ORDER BY author_name
            """,
            con=connection_string)

,author_name
0,Arthur Conan Doyle
1,Harper Lee
2,J.R.R. Tolkien


---
## 7.&nbsp; Preparing and sending the books table 📚
By extracting the authors table from our SQL database, we gain access to the unique identifier `author_id` assigned to each author. These `author_id`'s serve as pointers to their corresponding author records, allowing us to seamlessly link the `author_id`'s in the books table to their respective authors in the authors table, thereby completing the books table.

In [35]:
non_relational_df

,author_name,book_title,year_published
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887
1,J.R.R. Tolkien,The Hobbit,1937
2,J.R.R. Tolkien,The Lord of the Rings,1954
3,Harper Lee,To Kill a Mockingbird,1960
4,Harper Lee,Go Set a Watchman,2015


In [36]:
books_df = non_relational_df.merge(authors_from_sql,
                                   on = "author_name",
                                   how="left")

books_df

,author_name,book_title,year_published
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887
1,J.R.R. Tolkien,The Hobbit,1937
2,J.R.R. Tolkien,The Lord of the Rings,1954
3,Harper Lee,To Kill a Mockingbird,1960
4,Harper Lee,Go Set a Watchman,2015


In [37]:
books_df = books_df.drop(columns=["author_name"])

books_df

,book_title,year_published
0,The Adventures of Sherlock Holmes,1887
1,The Hobbit,1937
2,The Lord of the Rings,1954
3,To Kill a Mockingbird,1960
4,Go Set a Watchman,2015


In [38]:
books_df.to_sql('books',
                if_exists='append',
                con=connection_string,
                index=False)

5

In [39]:
books_from_sql = pd.read_sql("books", con=connection_string)
books_from_sql

,book_title,year_published
0,The Adventures of Sherlock Holmes,1887
1,The Hobbit,1937
2,The Lord of the Rings,1954
3,To Kill a Mockingbird,1960
4,Go Set a Watchman,2015


---
## 8.&nbsp; Challenge 😃
Now that you've learnt how to send and retrieve information, it's your turn to show off your skills. Create multiple tables in SQL for the data you scrapped about cities from Wikipedia. One should just be a table about the cities, the others should be facts about the cities.

| city_id | city |
| --- | --- |
| 1 | Berlin |
| 2 | Hamburg |
| 3 | Munich |

<br>

| City ID | Population | Year Data Retrieved |
|---|---|---|
| 1 | 3,850,809 | 2024 |
| 2 | 1,945,532 | 2024 |
| 3 | 1,512,491 | 2024 |

> **Pro Tip:** Visualise your relational database with pen and paper before you start coding. This can help you to identify any potential problems or inconsistencies in your design, and it can also make the coding process more efficient.

To write the web-scraped data to a SQL database, we first need to recreate it. That process was run in another notebook, since closed, so those DataFrames now need to be rebuilt before we can use them again.

In [40]:
import pandas as pd
import requests
import getpass
from bs4 import BeautifulSoup

In [42]:
schema = "gans"      # make sure to write to the correct schema
host = "127.0.0.1"
user = "root"
password = getpass.getpass("Enter password")  # getpass is fine for developing, but I'll use a different method, .env, later
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

Enter password ········


In [43]:
# copy working code from `01_webScraping__structure.ipynb` to get the city info dataframe

countries = []
latitudes = []
longitudes = []

cities = ["Berlin", "Hamburg", "Munich"]

headers = {'User-Agent': 'Chrome/134.0.0.0'}
for city in cities:
    url = f"https://en.wikipedia.org/wiki/{city}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        city_soup = BeautifulSoup(response.content, 'html.parser')
        for row in city_soup.find("table").find_all("tr"):
            if row.find(string="Country"): 
                countries.append(row.find("a").get_text())
        lat, lon = city_soup.find("table").find(class_="geo").get_text().split("; ")
        try:
            latitudes.append(float(lat))
            longitudes.append(float(lon))
        except ValueError:
            latitudes.append(None)
            longitudes.append(None)
            
    else:
        print(f"WARNING: Could not retrieve HTML for {city}")

cities_df = pd.DataFrame({"name": cities, "country": countries, "latitude": latitudes, "longitude": longitudes})
cities_df

,name,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


Now go to `03_gans_schema.sql` and write out the `CREATE TABLE` statement to match `cities_df` before using `.to_sql()` to write the DataFrame to the `gans` database.

In [44]:
cities_df.to_sql(
    "cities",
    con=connection_string,
    if_exists="append",
    index=False
)

3

In order to set up a foreign key, we need to read the cities from the SQL database, where `"city_id"` is assigned to each row.

In [45]:
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,name,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


In [46]:
# copy more code from `01_webScraping__structure.ipynb`, but modify to include foreign key

populations = []
city_ids = []     # add a list to keep foreign key values

# since we're pulling TWO values from each row ('city_id' and 'name'), we can't just loop through one column
# .iterrows() lets us loop through a DataFrame row by row
# it returns one index (`i`) and one horizontal Series (`row`) at a time
for _, row in cities_df.iterrows():  
    city = row["name"]
    city_id = row["city_id"]
    
    url = f"https://en.wikipedia.org/wiki/{city}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        city_soup = BeautifulSoup(response.content, 'html.parser')
        for row in city_soup.find("table").find_all("tr"):
            if row.find(string="Population"): 
                population = int(row.find_next("td").get_text().replace(",", ""))
    populations.append(population)
    city_ids.append(city_id)

pop_df = pd.DataFrame({"city_id": city_ids, "population": populations, "date_gathered": pd.Timestamp.now().date()})
pop_df

,city_id,population,date_gathered
0,1,3596999,2026-08-24
1,2,1973896,2026-08-24
2,3,1505005,2026-08-24


Before using `pop_df.to_sql()` to write the DataFrame to the `gans` database, go back to `03_gans_schema.sql` and write out the appropriate `CREATE TABLE` statement.

In [47]:
pop_df.to_sql(
    "populations",
    con=connection_string,
    if_exists="append",
    index=False
)

3